<a id="perceptron"></a>
# The single perceptron

$$z = \sum_i w_i x_i + b \qquad y = \begin{cases} 1 & z > 0 \\ 0 & \text{otherwise}\end{cases}$$

A weighted sum, then a threshold. Nothing else.


In [ ]:
def step(z):
    return 1 if z > 0 else 0


def perceptron(x, w, b):
    z = b                       # the bias: the vote cast before any input arrives
    for xi, wi in zip(x, w):
        z += wi * xi            # each input votes, scaled by how much it matters
    return step(z)


w, b = [1.0, 1.0], -1.5         # chosen by hand, not learned. this happens to be AND.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

CORNERS = [[0, 0], [0, 1], [1, 0], [1, 1]]


def show(w, b, want=None, title="", ax=None):
    """Draw what the neuron does to every point in the plane.

    want : the target output per corner. Any corner the neuron gets wrong
           is ringed in red.
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(4, 4))

    g = np.linspace(-0.6, 1.6, 400)
    X, Y = np.meshgrid(g, g)
    Z = w[0] * X + w[1] * Y + b                       # z, everywhere at once

    ax.contourf(X, Y, (Z > 0).astype(float), levels=[-.5, .5, 1.5],
                colors=["#e8eef7", "#b9d0ee"])        # the two half-planes
    ax.contour(X, Y, Z, levels=[0], colors="#1f4e8c", linewidths=2)   # z = 0

    for i, x in enumerate(CORNERS):
        fires = perceptron(x, w, b)
        wrong = want is not None and fires != want[i]
        ax.scatter(*x, s=300, zorder=3, linewidth=3,
                   edgecolor="#c1440e" if wrong else "#1f4e8c",
                   color="#1f4e8c" if fires else "white")
        ax.annotate(fires, x, color="white" if fires else "#1f4e8c",
                    ha="center", va="center", zorder=4, weight="bold")

    wv = np.array(w, float)
    foot = -b * wv / (wv @ wv)                        # point on the line nearest the origin
    ax.arrow(*foot, *(wv / np.linalg.norm(wv) * .35), width=.025, color="#c1440e",
             length_includes_head=True, zorder=5)     # w points into the firing side

    ax.set(xlim=(-.6, 1.6), ylim=(-.6, 1.6), xticks=[0, 1], yticks=[0, 1], title=title)
    ax.set_aspect("equal")
    return ax


show(w, b, want=[0, 0, 0, 1], title="AND")
plt.show()


In [ ]:
# Only the bias moved. AND demands 2 votes, OR is satisfied by 1.
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
show([1., 1.], -1.5, want=[0, 0, 0, 1], title="AND   b = -1.5", ax=axes[0])
show([1., 1.], -0.5, want=[0, 1, 1, 1], title="OR    b = -0.5", ax=axes[1])

# XOR: fire when the inputs DISAGREE. Best line found by brute force below.
show([1., 1.], -0.5, want=[0, 1, 1, 0], title="XOR   ...best effort", ax=axes[2])
plt.show()


In [ ]:
# Is that XOR line just a bad guess? Try every line. (Coarse, but the answer is stark.)
grid = np.linspace(-3, 3, 61)
best = {}
for name, want in [("AND", [0, 0, 0, 1]), ("OR", [0, 1, 1, 1]), ("XOR", [0, 1, 1, 0])]:
    scores = [sum(perceptron(x, [w1, w2], b) == t for x, t in zip(CORNERS, want))
              for w1 in grid for w2 in grid for b in grid]
    best[name] = np.bincount(scores, minlength=5)     # how many lines score 0,1,2,3,4

fig, ax = plt.subplots(figsize=(6, 3.5))
for i, (name, counts) in enumerate(best.items()):
    frac = counts / counts.sum()
    ax.bar(np.arange(5) + (i - 1) * .27, frac, width=.26, label=name,
           color=["#1f4e8c", "#7aa6d8", "#c1440e"][i])
ax.set(xticks=range(5), xlabel="corners correct (out of 4)", ylabel="fraction of lines",
       title="every line, scored")
ax.legend()
plt.show()
